In [1]:
# ============================================================
# COVID-19 Global Trends Analysis
# Notebook 2: SQL Analysis with PostgreSQL
# Author  : Udit Jadli
# Date    : March 2026
# ============================================================


# ============================================================
# SECTION 6 - SQL ANALYSIS
# ============================================================


# ---- 6.1 Setup PostgreSQL Connection ----

# These libraries help us talk to PostgreSQL from Python
import psycopg2                        # used to connect to PostgreSQL
from sqlalchemy import create_engine   # used to load our data into PostgreSQL tables
import pandas as pd                    # used to read our cleaned CSV files

# Fill in the connection details for our local PostgreSQL database
DB_NAME     = 'covid19_db'   # the database we created earlier in pgAdmin
DB_USER     = 'postgres'     # default username for PostgreSQL
DB_PASSWORD = 'admin123'     # password we set in pgAdmin
DB_HOST     = 'localhost'    # our database is running on this same computer
DB_PORT     = '5432'         # PostgreSQL always runs on this port by default

# Build the connection string and create an engine
# Think of the engine as a bridge between Python and our PostgreSQL database
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

# Try connecting to make sure everything is working before we load any data
with engine.connect() as connection:
    print(" Connected to PostgreSQL successfully!")
    print(f"   Database : {DB_NAME}")
    print(f"   Host     : {DB_HOST}")
    print(f"   Port     : {DB_PORT}")

 Connected to PostgreSQL successfully!
   Database : covid19_db
   Host     : localhost
   Port     : 5432


In [2]:
# ---- 6.2 Load Cleaned Data into PostgreSQL ----

import os

# Path to our cleaned CSV files
PROCESSED_PATH = '../data/processed/'

# List of cleaned files we want to load into PostgreSQL
# Each file will become a table in our covid19_db database
tables = {
    'country_wise'  : 'country_wise_latest_cleaned.csv',
    'covid_clean'   : 'covid_19_clean_complete_cleaned.csv',
    'day_wise'      : 'day_wise_cleaned.csv',
    'full_grouped'  : 'full_grouped_cleaned.csv',
    'worldometer'   : 'worldometer_data_cleaned.csv'
}

# Loop through each file and load it into PostgreSQL as a table
for table_name, file_name in tables.items():
    # Read the cleaned CSV file into a dataframe
    df = pd.read_csv(f'{PROCESSED_PATH}{file_name}')

    # Load the dataframe into PostgreSQL as a table
    # if_exists='replace' means if the table already exists, replace it
    # index=False means don't add an extra index column
    df.to_sql(table_name, engine, if_exists='replace', index=False)

    print(f" {table_name} loaded into PostgreSQL ({len(df)} rows)")

print("\n All tables loaded into PostgreSQL successfully!")

 country_wise loaded into PostgreSQL (187 rows)
 covid_clean loaded into PostgreSQL (49068 rows)
 day_wise loaded into PostgreSQL (188 rows)
 full_grouped loaded into PostgreSQL (35156 rows)
 worldometer loaded into PostgreSQL (209 rows)

 All tables loaded into PostgreSQL successfully!


In [4]:
# ---- 6.3 Basic SQL Queries ----

# Helper function to run any SQL query and return results as a dataframe
def run_query(query, description):
    # Print the query description so we know what we are looking at
    print(f"\n{'='*55}")
    print(f" {description}")
    print(f"{'='*55}")
    # Run the query and load results into a dataframe
    result = pd.read_sql(query, engine)
    print(result.to_string(index=False))
    return result


# Query 1 - Total confirmed, deaths and recovered globally
run_query("""
    SELECT 
        MAX("Confirmed")  AS total_confirmed,
        MAX("Deaths")     AS total_deaths,
        MAX("Recovered")  AS total_recovered
    FROM day_wise;
""", "Query 1 - Global Totals")


# Query 2 - Top 10 countries by confirmed cases
run_query("""
    SELECT 
        "Country/Region"    AS country,
        "Confirmed"         AS confirmed_cases,
        "Deaths"            AS deaths,
        "Recovered"         AS recovered
    FROM country_wise
    ORDER BY "Confirmed" DESC
    LIMIT 10;
""", "Query 2 - Top 10 Countries by Confirmed Cases")


# Query 3 - Top 10 countries by death rate
run_query("""
    SELECT 
        "Country/Region"            AS country,
        "Deaths / 100 Cases"        AS death_rate,
        "Recovered / 100 Cases"     AS recovery_rate
    FROM country_wise
    ORDER BY "Deaths / 100 Cases" DESC
    LIMIT 10;
""", "Query 3 - Top 10 Countries by Death Rate")


 Query 1 - Global Totals
 total_confirmed  total_deaths  total_recovered
        16480485        654036          9468087

 Query 2 - Top 10 Countries by Confirmed Cases
       country  confirmed_cases  deaths  recovered
            US          4290259  148011    1325804
        Brazil          2442375   87618    1846641
         India          1480073   33408     951166
        Russia           816680   13334     602249
  South Africa           452529    7067     274925
        Mexico           395489   44022     303810
          Peru           389717   18418     272547
         Chile           347923    9187     319954
United Kingdom           301708   45844       1437
          Iran           293606   15912     255144

 Query 3 - Top 10 Countries by Death Rate
       country  death_rate  recovery_rate
         Yemen       28.56          49.26
United Kingdom       15.19           0.48
       Belgium       14.79          26.27
         Italy       14.26          80.64
        France  

,country,death_rate,recovery_rate
0,Yemen,28.56,49.26
1,United Kingdom,15.19,0.48
2,Belgium,14.79,26.27
3,Italy,14.26,80.64
4,France,13.71,36.86
5,Hungary,13.40,74.84
6,Netherlands,11.53,0.35
7,Mexico,11.13,76.82
8,Spain,10.44,55.20
9,Western Sahara,10.00,80.00


In [11]:
# ---- 6.4 Intermediate SQL Queries ----

# Query 4 - Total cases grouped by WHO Region
run_query("""
    SELECT 
        "WHO Region"                AS who_region,
        SUM("Confirmed")            AS total_confirmed,
        SUM("Deaths")               AS total_deaths,
        SUM("Recovered")            AS total_recovered
    FROM country_wise
    GROUP BY "WHO Region"
    ORDER BY total_confirmed DESC;
""", "Query 4 - Total Cases by WHO Region")


# Query 5 - Countries with recovery rate above 70%
run_query("""
    SELECT 
        "Country/Region"            AS country,
        "Recovered / 100 Cases"     AS recovery_rate,
        "Deaths / 100 Cases"        AS death_rate,
        "Confirmed"                 AS confirmed_cases
    FROM country_wise
    WHERE "Recovered / 100 Cases" > 70
    ORDER BY recovery_rate DESC;
""", "Query 5 - Countries with Recovery Rate Above 70%")


# Query 6 - Countries with death rate above 10%
run_query("""
    SELECT 
        "Country/Region"            AS country,
        "Deaths / 100 Cases"        AS death_rate,
        "Confirmed"                 AS confirmed_cases,
        "Deaths"                    AS total_deaths
    FROM country_wise
    WHERE "Deaths / 100 Cases" > 10
    ORDER BY death_rate DESC;
""", "Query 6 - Countries with Death Rate Above 10%")


 Query 4 - Total Cases by WHO Region
           who_region  total_confirmed  total_deaths  total_recovered
             Americas        8839286.0      342732.0        4468616.0
               Europe        3299523.0      211144.0        1993723.0
      South-East Asia        1835297.0       41349.0        1156933.0
Eastern Mediterranean        1490744.0       38339.0        1201400.0
               Africa         723207.0       12223.0         440645.0
      Western Pacific         292428.0        8249.0         206770.0

 Query 5 - Countries with Recovery Rate Above 70%
                         country  recovery_rate  death_rate  confirmed_cases
                        Holy See         100.00        0.00               12
                        Dominica         100.00        0.00               18
                         Grenada         100.00        0.00               23
                        Djibouti          98.38        1.15             5059
                         Iceland    

,country,death_rate,confirmed_cases,total_deaths
0,Yemen,28.56,1691,483
1,United Kingdom,15.19,301708,45844
2,Belgium,14.79,66428,9822
3,Italy,14.26,246286,35112
4,France,13.71,220352,30212
5,Hungary,13.40,4448,596
6,Netherlands,11.53,53413,6160
7,Mexico,11.13,395489,44022
8,Spain,10.44,272421,28432


In [6]:
# ---- 6.5 Advanced SQL Queries ----

# Query 7 - Rank countries by confirmed cases using RANK() window function
# RANK() assigns a rank number to each country based on confirmed cases
run_query("""
    SELECT 
        "Country/Region"        AS country,
        "Confirmed"             AS confirmed_cases,
        RANK() OVER (
            ORDER BY "Confirmed" DESC
        )                       AS rank
    FROM country_wise
    LIMIT 10;
""", "Query 7 - Country Rankings by Confirmed Cases (RANK)")


# Query 8 - Using CTE to find countries above average confirmed cases
# CTE (Common Table Expression) is like a temporary table we create first
# then query from it
run_query("""
    WITH avg_cases AS (
        -- First calculate the average confirmed cases across all countries
        SELECT AVG("Confirmed") AS avg_confirmed
        FROM country_wise
    )
    -- Then find all countries above that average
    SELECT 
        "Country/Region"    AS country,
        "Confirmed"         AS confirmed_cases
    FROM country_wise, avg_cases
    WHERE "Confirmed" > avg_confirmed
    ORDER BY confirmed_cases DESC;
""", "Query 8 - Countries Above Average Confirmed Cases (CTE)")


# Query 9 - Daily new cases rolling 7 day average
# This smooths out daily spikes to show the real trend
run_query("""
    SELECT 
        "Date"                          AS date,
        "New cases"                     AS new_cases,
        ROUND(AVG("New cases") OVER (
            ORDER BY "Date"
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 0)                           AS rolling_7day_avg
    FROM day_wise
    ORDER BY "Date"
    LIMIT 20;
""", "Query 9 - Rolling 7-Day Average of New Cases")


 Query 7 - Country Rankings by Confirmed Cases (RANK)
       country  confirmed_cases  rank
            US          4290259     1
        Brazil          2442375     2
         India          1480073     3
        Russia           816680     4
  South Africa           452529     5
        Mexico           395489     6
          Peru           389717     7
         Chile           347923     8
United Kingdom           301708     9
          Iran           293606    10

 Query 8 - Countries Above Average Confirmed Cases (CTE)
       country  confirmed_cases
            US          4290259
        Brazil          2442375
         India          1480073
        Russia           816680
  South Africa           452529
        Mexico           395489
          Peru           389717
         Chile           347923
United Kingdom           301708
          Iran           293606
      Pakistan           274289
         Spain           272421
  Saudi Arabia           268934
      Colombia       

,date,new_cases,rolling_7day_avg
0,2020-01-22,0,0.0
1,2020-01-23,99,50.0
2,2020-01-24,287,129.0
3,2020-01-25,493,220.0
4,2020-01-26,684,313.0
5,2020-01-27,809,395.0
6,2020-01-28,2651,718.0
7,2020-01-29,588,802.0
8,2020-01-30,2068,1083.0
9,2020-01-31,1693,1284.0
